In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score


In [2]:
data = pd.read_csv(r'C:\Users\Vineet\OneDrive\Desktop\credit_card_resume\data\creditcard.csv')
X = data.drop(columns="Class")
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [3]:

def build_model(in_dim, hidden, dropout):
    layers, d = [], in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
        d = h
    layers.append(nn.Linear(d, 1))
    return nn.Sequential(*layers)


def train_nn(X_t, y_np, cfg, seed):
    torch.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_t, torch.tensor(y_np)),
        batch_size=cfg["batch_size"], shuffle=True,
        generator=torch.Generator().manual_seed(seed))

    model = build_model(X_t.shape[1], cfg["hidden"], cfg["dropout"])

    pw = (y_np == 0).sum() / (y_np == 1).sum()      # from training labels only
    pw = {None: 1.0, "sqrt": pw ** 0.5, "full": pw}[cfg["pw_mode"]]
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pw)))
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"],
                           weight_decay=cfg["weight_decay"])

    for _ in range(cfg["epochs"]):
        model.train()
        for xb, yb in loader:
            loss = criterion(model(xb).flatten(), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


def predict_proba(model, X_t):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X_t).flatten()).numpy()


def cv_nn(cfg):
    scores = []
    for fold, (tr, va) in enumerate(cv.split(X_train, y_train), start=1):
        Xtr, Xva = X_train.iloc[tr].copy(), X_train.iloc[va].copy()
        ytr = y_train.iloc[tr].to_numpy(dtype=np.float32)
        yva = y_train.iloc[va].to_numpy().astype(int)

        sc = StandardScaler()                         # fit inside each fold
        Xtr[["Time", "Amount"]] = sc.fit_transform(Xtr[["Time", "Amount"]])
        Xva[["Time", "Amount"]] = sc.transform(Xva[["Time", "Amount"]])

        Xtr_t = torch.tensor(Xtr.to_numpy(dtype=np.float32))
        Xva_t = torch.tensor(Xva.to_numpy(dtype=np.float32))

        model = train_nn(Xtr_t, ytr, cfg, seed=42 + fold)
        scores.append(average_precision_score(yva, predict_proba(model, Xva_t)))
    return float(np.mean(scores)), float(np.std(scores))

In [4]:

# Random search (same budget as the RF search: 4 configs x 5 folds = 20 fits)
nn_grid = {
    "hidden": [(32,), (64, 32), (128, 64)],
    "dropout": [0.0, 0.2],
    "lr": [1e-3, 3e-3],
    "weight_decay": [0, 1e-5],
    "epochs": [30, 50],
    "batch_size": [1000, 2000],
    "pw_mode": [None, "sqrt"],
}

N_ITER = 4
results = []
for i, cfg in enumerate(ParameterSampler(nn_grid, n_iter=N_ITER, random_state=42), 1):
    m, s = cv_nn(cfg)
    results.append({"cfg": cfg, "pr_auc": m, "std": s})
    print(f"{i}/{N_ITER}  PR-AUC {m:.4f} ± {s:.4f}  {cfg}")



1/4  PR-AUC 0.8199 ± 0.0174  {'weight_decay': 1e-05, 'pw_mode': None, 'lr': 0.003, 'hidden': (128, 64), 'epochs': 50, 'dropout': 0.0, 'batch_size': 1000}
2/4  PR-AUC 0.8243 ± 0.0187  {'weight_decay': 0, 'pw_mode': None, 'lr': 0.001, 'hidden': (128, 64), 'epochs': 50, 'dropout': 0.0, 'batch_size': 2000}
3/4  PR-AUC 0.8427 ± 0.0268  {'weight_decay': 0, 'pw_mode': None, 'lr': 0.003, 'hidden': (32,), 'epochs': 50, 'dropout': 0.2, 'batch_size': 1000}
4/4  PR-AUC 0.8162 ± 0.0285  {'weight_decay': 1e-05, 'pw_mode': 'sqrt', 'lr': 0.003, 'hidden': (128, 64), 'epochs': 50, 'dropout': 0.0, 'batch_size': 2000}


In [5]:
best = max(results, key=lambda r: r["pr_auc"])
best_cfg = best["cfg"]

print("\nBest config:", best_cfg)
print(f"Best CV PR-AUC: {best['pr_auc']:.4f} ± {best['std']:.4f}")

# Refit best config on the full X_train (scaler fit on X_train only)
scaler = StandardScaler()
X_tr_full = X_train.copy()
X_tr_full[["Time", "Amount"]] = scaler.fit_transform(X_tr_full[["Time", "Amount"]])
Xtr_t = torch.tensor(X_tr_full.to_numpy(dtype=np.float32))

best_nn = train_nn(Xtr_t, y_train.to_numpy(dtype=np.float32), best_cfg, seed=42)


Best config: {'weight_decay': 0, 'pw_mode': None, 'lr': 0.003, 'hidden': (32,), 'epochs': 50, 'dropout': 0.2, 'batch_size': 1000}
Best CV PR-AUC: 0.8427 ± 0.0268


In [6]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score, confusion_matrix
)


X_te = X_test.copy()
X_te[["Time", "Amount"]] = scaler.transform(X_te[["Time", "Amount"]])
X_te_t = torch.tensor(X_te.to_numpy(dtype=np.float32))

probs = predict_proba(best_nn, X_te_t)
preds = (probs >= 0.5).astype(int)

print(f"PR-AUC    : {average_precision_score(y_test, probs):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, probs):.4f}")
print(f"Precision : {precision_score(y_test, preds):.4f}")
print(f"Recall    : {recall_score(y_test, preds):.4f}")
print(f"F1        : {f1_score(y_test, preds):.4f}")

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

PR-AUC    : 0.8475
ROC-AUC   : 0.9858
Precision : 0.8837
Recall    : 0.7755
F1        : 0.8261

TN=56854  FP=10  FN=22  TP=76
